In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.mask import mask
from shapely.geometry import box, mapping

# === Region mapping ===
state_to_region_division = {
    'Connecticut': ('Northeast', 'New England'), 'Maine': ('Northeast', 'New England'),
    'Massachusetts': ('Northeast', 'New England'), 'New Hampshire': ('Northeast', 'New England'),
    'Rhode Island': ('Northeast', 'New England'), 'Vermont': ('Northeast', 'New England'),
    'New Jersey': ('Northeast', 'Middle Atlantic'), 'New York': ('Northeast', 'Middle Atlantic'),
    'Pennsylvania': ('Northeast', 'Middle Atlantic'),
    'Illinois': ('Midwest', 'East North Central'), 'Indiana': ('Midwest', 'East North Central'),
    'Michigan': ('Midwest', 'East North Central'), 'Ohio': ('Midwest', 'East North Central'),
    'Wisconsin': ('Midwest', 'East North Central'),
    'Iowa': ('Midwest', 'West North Central'), 'Kansas': ('Midwest', 'West North Central'),
    'Minnesota': ('Midwest', 'West North Central'), 'Missouri': ('Midwest', 'West North Central'),
    'Nebraska': ('Midwest', 'West North Central'), 'North Dakota': ('Midwest', 'West North Central'),
    'South Dakota': ('Midwest', 'West North Central'),
    'Delaware': ('South', 'South Atlantic'), 'District of Columbia': ('South', 'South Atlantic'),
    'Florida': ('South', 'South Atlantic'), 'Georgia': ('South', 'South Atlantic'),
    'Maryland': ('South', 'South Atlantic'), 'North Carolina': ('South', 'South Atlantic'),
    'South Carolina': ('South', 'South Atlantic'), 'Virginia': ('South', 'South Atlantic'),
    'West Virginia': ('South', 'South Atlantic'),
    'Alabama': ('South', 'East South Central'), 'Kentucky': ('South', 'East South Central'),
    'Mississippi': ('South', 'East South Central'), 'Tennessee': ('South', 'East South Central'),
    'Arkansas': ('South', 'West South Central'), 'Louisiana': ('South', 'West South Central'),
    'Oklahoma': ('South', 'West South Central'), 'Texas': ('South', 'West South Central'),
    'Arizona': ('West', 'Mountain'), 'Colorado': ('West', 'Mountain'), 'Idaho': ('West', 'Mountain'),
    'Montana': ('West', 'Mountain'), 'Nevada': ('West', 'Mountain'), 'New Mexico': ('West', 'Mountain'),
    'Utah': ('West', 'Mountain'), 'Wyoming': ('West', 'Mountain'),
    'Alaska': ('West', 'Pacific'), 'California': ('West', 'Pacific'),
    'Hawaii': ('West', 'Pacific'), 'Oregon': ('West', 'Pacific'), 'Washington': ('West', 'Pacific')
}

# === Paths ===
depth_tif_2021 = "/mnt/large_vol/hangkai/Forest_Depth_Classification/LCMAP_CU_2021_V13_LCPRI.tif"
sample_raster_path = depth_tif_2021
shapefile = "/mnt/large_vol/hangkai/shapefile/CONUS.shp"  # Assumes state names are in "NAME" column

# === Load raster extent from sample year ===
with rasterio.open(sample_raster_path) as sample_raster:
    raster_crs = sample_raster.crs
    bounds = sample_raster.bounds
    extent_geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)

# === Load and clip shapefile ===
conus_gdf = gpd.read_file(shapefile)

if conus_gdf.crs != raster_crs:
    conus_gdf = conus_gdf.to_crs(raster_crs)

conus_clipped = gpd.clip(conus_gdf, extent_geom)
conus_clipped = conus_clipped[["geometry", "NAME", "STUSPS"]]

# === Add Region info and filter ===
conus_clipped["Region"] = conus_clipped["NAME"].map(lambda x: state_to_region_division.get(x, (None, None))[0])
conus_clipped = conus_clipped[~conus_clipped["Region"].isna()]  # Drop unmatched states

# === Dissolve into regions ===
region_gdf = conus_clipped.dissolve(by="Region")

# === Forest area computation function ===
def compute_forest_areas(geometry, raster_path):
    with rasterio.open(raster_path) as src:
        data, _ = mask(src, [mapping(geometry)], crop=True, filled=True, nodata=0)
        data = data[0].astype(np.uint8)
        pixel_area = 30 * 30  # m²

        exterior_area = np.sum(np.isin(data, [1, 2, 3, 4])) * pixel_area
        total_area = np.sum(np.isin(data, [1, 2, 3, 4, 5])) * pixel_area

        if total_area == 0:
            return np.nan, 0, 0
        else:
            return exterior_area / total_area, exterior_area, total_area

# === Compute stats by region ===
print("===== 2021 Area-weighted Exterior Forest Rate by Region =====")
for region_name, row in region_gdf.iterrows():
    ratio, ext_area, tot_area = compute_forest_areas(row.geometry, depth_tif_2021)
    print(f"{region_name}: {ratio:.4f} (Exterior: {ext_area/1e6:.2f} km², Total: {tot_area/1e6:.2f} km²)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from matplotlib.colors import Normalize, to_rgba
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Patch
import geopandas as gpd
from shapely.geometry import MultiPolygon

# === Reprojection function ===
def reproject_to_wgs84(input_path, resampling=Resampling.bilinear):
    with rasterio.open(input_path) as src:
        dst_crs = "EPSG:4326"
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        dst_data = np.zeros((height, width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=resampling)
    return dst_data, transform, width, height

def get_extent(transform, width, height):
    xmin = transform[2]
    ymax = transform[5]
    xmax = xmin + width * transform[0]
    ymin = ymax + height * transform[4]
    return [xmin, xmax, ymin, ymax]

def base_map(ax):
    states = cfeature.NaturalEarthFeature(
        'cultural', 'admin_1_states_provinces_lines', '50m',
        edgecolor='gray', facecolor='none')
    ax.add_feature(cfeature.LAND, alpha=0.1)
    ax.add_feature(cfeature.COASTLINE, lw=0.2)
    ax.add_feature(cfeature.BORDERS, linestyle='--', lw=0.3, alpha=0.5)
    ax.add_feature(cfeature.RIVERS, lw=0.2)
    ax.add_feature(cfeature.LAKES, alpha=0.2)
    ax.add_feature(states, lw=0.2)
    gl = ax.gridlines(draw_labels=True, linestyle=":", linewidth=0.3, color="k")
    gl.top_labels = False
    gl.right_labels = False

def lighten_color(color, amount=0.5):
    c = np.array(to_rgba(color))
    return tuple(c[:3] + (1.0 - c[:3]) * amount) + (1.0,)

# === Region colors ===
base_colors = {
    "Midwest": "#d73027",     # red
    "Northeast": "#984ea3",   # purple
    "South": "#ffae00",       # yellow
    "West": "#1f78b4"         # blue
}

# === Load raster and reproject ===
exterior_path = "/mnt/large_vol/hangkai/Forest_Exterior_Forest_Ratio_1km/Exterior_Forest_Rate_2021_1km.tif"
data, transform, width, height = reproject_to_wgs84(exterior_path)
extent = get_extent(transform, width, height)
data = np.where((data > 0) & (data <= 1), data, np.nan)

# === Load new CONUS shapefile and assign region manually ===
shp_path = "/mnt/large_vol/hangkai/shapefile/cb_2018_us_state_5m.shp"
states_gdf = gpd.read_file(shp_path).to_crs("EPSG:4326")

# === Manual region mapping by NAME ===
state_to_region = {
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast',
    'New Hampshire': 'Northeast', 'Rhode Island': 'Northeast', 'Vermont': 'Northeast',
    'New Jersey': 'Northeast', 'New York': 'Northeast', 'Pennsylvania': 'Northeast',
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Michigan': 'Midwest',
    'Ohio': 'Midwest', 'Wisconsin': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest',
    'Minnesota': 'Midwest', 'Missouri': 'Midwest', 'Nebraska': 'Midwest',
    'North Dakota': 'Midwest', 'South Dakota': 'Midwest',
    'Delaware': 'South', 'District of Columbia': 'South', 'Florida': 'South',
    'Georgia': 'South', 'Maryland': 'South', 'North Carolina': 'South',
    'South Carolina': 'South', 'Virginia': 'South', 'West Virginia': 'South',
    'Alabama': 'South', 'Kentucky': 'South', 'Mississippi': 'South',
    'Tennessee': 'South', 'Arkansas': 'South', 'Louisiana': 'South',
    'Oklahoma': 'South', 'Texas': 'South',
    'Arizona': 'West', 'Colorado': 'West', 'Idaho': 'West', 'Montana': 'West',
    'Nevada': 'West', 'New Mexico': 'West', 'Utah': 'West', 'Wyoming': 'West',
    'Alaska': 'West', 'California': 'West', 'Hawaii': 'West',
    'Oregon': 'West', 'Washington': 'West'
}

states_gdf["Region"] = states_gdf["NAME"].map(state_to_region)
states_gdf = states_gdf[~states_gdf["Region"].isna()]  # keep only matched states

# === Dissolve by region and clean geometry ===
region_geoms = (
    states_gdf
    .groupby("Region")["geometry"]
    .apply(lambda x: x.unary_union)
    .reset_index()
)
region_gdf = gpd.GeoDataFrame(region_geoms, geometry="geometry", crs="EPSG:4326")
region_gdf["geometry"] = region_gdf["geometry"].buffer(0)  # fix invalid geometries

In [ ]:
# === Prepare figure and map ===
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.AlbersEqualArea(central_longitude=-96, central_latitude=23)}, dpi=600)
ax.set_extent([-125, -66.5, 24, 50], crs=ccrs.PlateCarree())
base_map(ax)

norm = Normalize(vmin=0, vmax=np.nanpercentile(data, 100))
im = ax.imshow(data, origin='upper', extent=extent, transform=ccrs.PlateCarree(),
               cmap='viridis', norm=norm)

# === Colorbar ===
cax = inset_axes(ax, width="30%", height="5%", loc='lower center',
                 bbox_to_anchor=(-0.28, 0.1, 1, 1), bbox_transform=ax.transAxes)
cbar = plt.colorbar(im, cax=cax, orientation='horizontal')
cbar.ax.xaxis.set_label_position('top')     # Correct: label on top for horizontal bar
cbar.ax.xaxis.set_ticks_position('bottom')     # Show ticks on top
cbar.set_label("Exterior Forest Fraction", fontsize=9, labelpad=5)

# === Draw region borders on the map ===
for _, row in region_gdf.iterrows():
    region_name = row["Region"]
    geom = row.geometry
    color = base_colors[region_name]
    if isinstance(geom, MultiPolygon):
        for part in geom.geoms:
            ax.add_geometries([part], crs=ccrs.PlateCarree(),
                              edgecolor=color, facecolor='none', linewidth=1.5, zorder=10)
    else:
        ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                          edgecolor=color, facecolor='none', linewidth=1.5, zorder=10)

# === Pie chart data ===
region_labels = ["Midwest", "Northeast", "South", "West"]
region_exteriors = [257387.8, 159359.02, 542298.26, 418333.93]
region_totals = [337300.11, 259998.14, 762744.53, 692299.21]
region_interior = [t - e for t, e in zip(region_totals, region_exteriors)]

pie_vals = []
pie_colors = []
for region, ext, intv in zip(region_labels, region_exteriors, region_interior):
    pie_vals.extend([ext, intv])
    pie_colors.append(base_colors[region])
    pie_colors.append(lighten_color(base_colors[region]))

# === Pie chart inset ===
pie_ax = inset_axes(ax, width="25%", height="25%", loc='lower left',
                    bbox_to_anchor=(0.76, 0.35, 1, 1), bbox_transform=ax.transAxes)

# Draw pie with percentages
wedges, texts, autotexts = pie_ax.pie(
    pie_vals,
    labels=None,
    colors=pie_colors,
    startangle=90,
    autopct='%1.0f%%',
    textprops={'fontsize': 6}
)

# Add legend with exterior and interior areas in Mha
legend_elements = []
for i, region in enumerate(region_labels):
    ext_patch = Patch(facecolor=base_colors[region],
                      label=f"{region} Exterior")
    int_patch = Patch(facecolor=lighten_color(base_colors[region]),
                      label=f"{region} Interior")
    legend_elements.extend([ext_patch, int_patch])

pie_ax.legend(handles=legend_elements, loc="center", bbox_to_anchor=(0.5, -0.45), fontsize=7, ncol=1)
pie_ax.set_title("Forest Composition\nby Region", fontsize=8, y=0.9)

# === Title and show ===
#ax.set_title("Exterior Forest Rate Distribution (2021)", fontsize=12, fontweight='bold', loc='left')
plt.tight_layout()
plt.savefig("/mnt/large_vol/hangkai/Forest_Exterior_Visualization_2021.png", dpi=600)
plt.show()